# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_after.png" style="width:300px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## Setup

Load and/or check for needed environmental variables

In [1]:
%%capture
%pip install --upgrade langgraph langchain langchain-ollama langchain-community
%pip install --upgrade python-dotenv

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Ollama server URL - override with OLLAMA_BASE_URL env var
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

In [3]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

C:\Users\vijay\AppData\Local\Temp\ipykernel_31068\3731249078.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [4]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [5]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [6]:
from sqlalchemy import inspect

# llama3.2 is too small to discover the schema by itself, so list every table and its columns.
inspector = inspect(db._engine)
SCHEMA = "\n".join(
    f"{table}({', '.join(col['name'] for col in inspector.get_columns(table))})"
    for table in db.get_usable_table_names()
)

SYSTEM_PROMPT = f"""You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
- Use only the tables and columns in this schema:
{SCHEMA}
"""

In [7]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

chat = ChatOllama(
    model="llama3.2:latest",
    temperature=0.7,
    base_url=OLLAMA_BASE_URL,
)

agent = create_agent(
    model=chat,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [8]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================

{"name":"execute_sql","parameters{"query":"SELECT Total FROM Invoice WHERE CustomerId = (SELECT CustomerId FROM Customer WHERE FirstName = 'Frank' AND LastName = 'Harris')"}}


In [9]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

{"name":"execute_sql","parameters\":{\"query\":\"SELECT Title FROM Album LIMIT 5\"}}


## Add memory

In [10]:
from langgraph.checkpoint.memory import InMemorySaver

In [11]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model=chat,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [12]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (6014255c-719c-4335-b26e-0197e39ee834)
 Call ID: 6014255c-719c-4335-b26e-0197e39ee834
  Args:
    query: SELECT Total FROM Invoice WHERE CustomerId = (SELECT CustomerId FROM Customer WHERE FirstName = "Frank" AND LastName = "Harris")
================================= Tool Message =================================
Name: execute_sql

[(0.99,), (1.98,), (13.86,), (8.91,), (1.98,), (3.96,), (5.94,)]
================================== Ai Message ==================================

The total on your last invoice was $0.99.


In [13]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (78c6ede8-f99d-4372-b97c-0af6086f7628)
 Call ID: 78c6ede8-f99d-4372-b97c-0af6086f7628
  Args:
    query: SELECT Title FROM Album
================================= Tool Message =================================
Name: execute_sql

[('For Those About To Rock We Salute You',), ('Balls to the Wall',), ('Restless and Wild',), ('Let There Be Rock',), ('Big Ones',), ('Jagged Little Pill',), ('Facelift',), ('Warner 25 Anos',), ('Plays Metallica By Four Cellos',), ('Audioslave',), ('Out Of Exile',), ('BackBeat Soundtrack',), ('The Best Of Billy Cobham',), ('Alcohol Fueled Brewtality Live! [Disc 1]',), ('Alcohol Fueled Brewtality Live! [Disc 2]',), ('Black Sabbath',), ('Black Sabbath Vol. 4 (Remaster)',), ('Body Count',), ('Chemical Wedding',), ('The Best Of Buddy Guy - The Millenium Colle

## Try your own queries
Now that there is memory, check the agents recall!

In [14]:
question = "Your Question Here?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

Your Question Here?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (cfd2458e-fad7-405a-a61d-9f96d9cdc510)
 Call ID: cfd2458e-fad7-405a-a61d-9f96d9cdc510
  Args:
    query: SELECT Title FROM Album
================================= Tool Message =================================
Name: execute_sql

[('For Those About To Rock We Salute You',), ('Balls to the Wall',), ('Restless and Wild',), ('Let There Be Rock',), ('Big Ones',), ('Jagged Little Pill',), ('Facelift',), ('Warner 25 Anos',), ('Plays Metallica By Four Cellos',), ('Audioslave',), ('Out Of Exile',), ('BackBeat Soundtrack',), ('The Best Of Billy Cobham',), ('Alcohol Fueled Brewtality Live! [Disc 1]',), ('Alcohol Fueled Brewtality Live! [Disc 2]',), ('Black Sabbath',), ('Black Sabbath Vol. 4 (Remaster)',), ('Body Count',), ('Chemical Wedding',), ('The Best Of Buddy Guy - The Millenium Collect